# Calculate mortality at each grid point using central estimates only

$M(x, y) = POP(x, y) \; \times \; BMR_c \; \times \; AF(x, y)  $

This script calculates gridpoint mortality for each mortality health outcome. The final step calculates the sum of the 17 outcomes to estimate total temperature mortality

In [ ]:
import os
import glob
import json
import numpy as np
import xarray as xr
from utils.mortality_utils import mortality
from utils.utils import get_scenario_config
import config
from utils.utils import require_dir
import pathlib

In [ ]:
# === Path config ===
BMR_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "BMR")
POP_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "SSP_pop" / "SSP2")
ID_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "GBD23")
MASKS_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "BMR" / "masks" / "country", "country masks")

In [ ]:
# Load country id mask
mask_file = "country_id_mask.nc"
mask_path = os.path.join(MASKS_DIR, mask_file)
masks = xr.open_dataarray(mask_path)

# Load population file
pop_file = "ssp2_coarse_grid_annual_2000-2100.nc"
pop_path = os.path.join(POP_DIR, pop_file)
population = xr.open_dataarray(pop_path)
pop = population.reindex_like(masks, method="nearest", tolerance=1e-9)

# Load look up file
lookup_file = "country_location_ids.json"
lookup_path = os.path.join(ID_DIR, lookup_file)
with open(lookup_path, "r", encoding="utf-8") as f:
    country_to_id = json.load(f)

In [ ]:
# === Health variables ===
health_vars = ["CHRONIC_KIDNEY_DISEASE", "CARDIOMYOPATHY_MYOCARDITIS",
               "HYPERTENSIVE_HEART_DISEASE", "ISCHEMIC_HEART_DISEASE",
               "STROKE", "DIABETES", "INJ_ANIMAL", "INJ_DISASTER",
               "DROWNING", "HOMICIDE", "MECHANICAL_INJ",
               "OTHER_UNINTENTIONAL_INJ", "SUICIDE", "OTHER_TRANSPORT_INJ",
               "ROAD_TRANSPORT_INJ", "LOWER_RESPIRATORY_INFECTIONS", "COPD"]

In [ ]:
# === Scenario and path config ===
# Set to whatever scenario and model you want
# Function returns error if not recognised
model = "CESM2"
scenario = "SSP245"
# For RR curves and file name
GBD_version = "GBD23"

configs = get_scenario_config(model, scenario)
ensemble_members = configs["ensemble_members"]
years = configs["years"]
dates = f"{years.start}-{years.stop}"

RR_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / GBD_version / "RR_curves")
TEMP_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / model / "temperature")
SAVE_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / model / "mortality" / "temperature" / "gridpoint_mortality")

# === Main loop ===
for health_VAR in health_vars:
    print(f"Processing mortality outcome {health_VAR}")

    for name, cid in country_to_id.items():
        print(f"Processing country ID {cid}, {name}")

        logRR_file = f"{health_VAR}_norm_logRR_{cid}.nc"
        logRR_path = os.path.join(RR_DIR, logRR_file)
        logRR = xr.open_dataarray(logRR_path)

    bmr_file = f"{GBD_version}_BMR_Global_Country_Map_{health_VAR}_1990-2009.nc"
    bmr_path = os.path.join(BMR_DIR, bmr_file)
    BMR = xr.open_dataarray(bmr_path)  # central estimate

    for ens_num in ensemble_members:
        print(f"Processing {scenario}, Ensemble {ens_num:02d}, {health_VAR}")

        pm25_file = f"Annual_PM25_BC_{model}_{scenario}_{ens_num:02d}_{dates}.nc"
        pm25_path = os.path.join(PM25_DIR, pm25_file)
        pm25 = xr.open_dataarray(pm25_path)

        # Adjust indices to match (with small tolerance)
        # e.g., max 1e-7 km distance
        pm25 = pm25.reindex_like(BMR, method="nearest", tolerance=1e-2, fill_value=0)

        M = []

        print(f"Processing years {years.start}-{years.stop}")
        for year in range(years.start, years.stop + 1):
            POP = pop.sel(year=year)

            # Find RR at each grid point
            RR = scaled_RR.interp(exposure=pm25.sel(year=year))
            # Calculate the attributable fraction
            AF = 1 - (1/RR)

            del RR

            # Calculate mortality at each grid point
            mortality_year = mortality(AF, BMR, POP)
            M.append(mortality_year)

            del mortality_year, AF, POP

        M_cleaned = [da.drop_vars("year", errors="ignore") for da in M]

        del M

        mortality_timeseries = xr.concat(
            M_cleaned,
            dim=(xr.DataArray(pm25["year"].values,
                              dims="year", name="year"))
        )

        mortality_timeseries = mortality_timeseries.drop_vars(
            ["exposure", "quantile", "region"]
        )

        del M_cleaned

        description = (f"Total {health_VAR} mortality due to PM2.5 using "
                       "central estimates only - scripts "
                       "by A.F. Wells (2025)")
        mortality_timeseries.attrs["description"] = description
        mortality_timeseries.attrs["GBD version"] = GBD_version
        mortality_timeseries.attrs["health_var"] = health_VAR
        mortality_timeseries.attrs["ensemble_number"] = ens_num
        mortality_timeseries.attrs["scenario"] = scenario
        mortality_timeseries.attrs["model"] = model

        out_file = f"Mortality_{GBD_version}_{health_VAR}_{model}_{scenario}_{ens_num:02d}_{dates}.nc"
        out_path = os.path.join(SAVE_DIR, out_file)

        print(f"Saving mortality timeseries to {out_path}")
        mortality_timeseries.to_netcdf(out_path)
        mortality_timeseries.close()

        del mortality_timeseries

    del BMR

    del RR_values, logRR, logRR_shifted, scaled_RR

print("All processing complete.")

## Save the sum of all mortality outcomes

In [ ]:
for ens_num in ensemble_members:
    # Find all files for this ensemble
    in_files = f"Mortality_{GBD_version}_*_{model}_{scenario}_{ens_num:02d}_{dates}.nc"
    in_path = os.path.join(SAVE_DIR, in_files)
    files = sorted(glob.glob(in_path))

    # Open and combine
    datasets = [xr.open_dataarray(f) for f in files]

    # Align (important in case of slight coordinate mismatches)
    aligned = xr.align(*datasets, join="exact")

    # Sum across the health variables
    summed_da = sum(aligned)

    description = ("Total mortality due to PM2.5 using "
                   "central estimates only - scripts "
                   "by A.F. Wells (2025)")
    summed_da.attrs["description"] = description
    summed_da.attrs["GBD version"] = GBD_version
    summed_da.attrs["ensemble_number"] = ens_num
    summed_da.attrs["scenario"] = scenario
    summed_da.attrs["model"] = model

    out_file = f"Mortality_{GBD_version}_{model}_{scenario}_{ens_num:02d}_{dates}.nc"
    out_path = os.path.join(SAVE_DIR, out_file)

    print(f"Saving summed mortality timeseries to {out_path}")
    summed_da.to_netcdf(out_path)

print("All processing complete.")